# Music Genre Classification using Machine Learning, Deep Learning, and DSP

## Abstract

This project addresses supervised music genre classification by combining Digital Signal Processing (DSP), classical Machine Learning, Deep Learning, transfer learning, and deployment-oriented model optimization. The classification task is built from two complementary audio datasets: GTZAN, a compact benchmark organized by genre folders, and FMA Medium, a larger metadata-driven music collection. Because the datasets use different genre taxonomies, the project harmonizes them into seven shared labels: blues, classical, country, hiphop, jazz, pop, and rock. Raw audio is standardized through a reproducible preprocessing pipeline that resamples audio to 22,050 Hz, converts it to mono, center-crops recordings to 28 seconds, and avoids aggressive amplitude normalization in order to preserve musical dynamics.

Feature extraction is performed with DSP representations tailored to different modeling approaches. MFCC features are summarized using per-coefficient means and standard deviations, producing compact 40-dimensional vectors for classical models. Mel Spectrogram tensors preserve richer time-frequency information and are used for neural-network experiments. Classical baselines include SVM and Random Forest models, with validation macro F1 used for model selection because the combined dataset is strongly imbalanced. The SVM trained on MFCC features provides the strongest overall macro F1 and demonstrates that compact DSP features remain effective for this task.

The Deep Learning experiments evaluate a small CNN trained from scratch and a MobileNetV2 transfer-learning model adapted to Mel Spectrogram inputs. MobileNetV2 improves over the CNN baseline in macro F1, but neither neural approach outperforms the MFCC-based SVM under the current dataset and imbalance conditions. The final stage evaluates post-training quantization of MobileNetV2 for deployment. Full static INT8 quantization reduces size and latency but severely harms macro F1, while dynamic Linear-only and classifier-only quantization preserve predictive performance with smaller deployment benefits. Overall, the project shows that model accuracy, class-balanced performance, and deployment efficiency must be evaluated together rather than optimized independently.

## Introduction

Music genre classification is an interesting Machine Learning problem because musical signals contain information across time, frequency, rhythm, timbre, instrumentation, and production style. Unlike simple tabular classification tasks, audio must first be transformed into numerical representations that expose meaningful structure for learning algorithms.

This project investigates both classical Machine Learning and Deep Learning because each approach uses a different view of the same audio data. Classical models can learn from compact MFCC summary vectors, which are efficient and interpretable as engineered DSP descriptors. Deep Learning models can operate on Mel Spectrogram tensors, preserving local time-frequency patterns that may be useful for convolutional architectures.

Deployment optimization is included because a useful model should not only perform well on test data, but also be practical to store and run. Post-training quantization is therefore evaluated on MobileNetV2 to measure the trade-off between model size, CPU inference speed, and predictive performance.

## Objectives

The main objectives of the project are:

- Explore GTZAN and FMA Medium audio data and metadata.
- Harmonize both datasets into a shared seven-genre classification task.
- Create reproducible train, validation, and test splits.
- Implement standardized audio preprocessing using reusable source modules.
- Extract MFCC features for classical Machine Learning models.
- Extract Mel Spectrogram tensors for CNN and transfer-learning models.
- Train and evaluate SVM and Random Forest baselines.
- Train and evaluate a CNN baseline on Mel Spectrograms.
- Train and evaluate MobileNetV2 transfer learning on Mel Spectrograms.
- Compare models using accuracy, macro F1, weighted F1, classification reports, and confusion matrices.
- Evaluate MobileNetV2 post-training quantization for deployment size and CPU inference speed.
- Preserve reproducibility by saving generated artifacts and centralizing configuration.

## Research Hypotheses

| ID | Evaluation-Aligned Hypothesis |
|---|---|
| H1 | Classical MFCC-based models are evaluated to determine whether compact DSP summary features can achieve useful validation macro F1 on the combined seven-genre dataset. A validation macro F1 threshold of 0.50 is used as the project criterion. |
| H2 | The CNN baseline is evaluated to determine whether a model trained directly on Mel Spectrogram tensors can achieve useful validation macro F1. A validation macro F1 threshold of 0.50 is used as the project criterion. |
| H3 | The CNN baseline is compared with the best classical model to assess whether learned Mel Spectrogram representations can match or exceed compact MFCC-based classical learning. The project considers a validation macro F1 difference of no more than 0.05 to indicate comparable performance. |
| H4 | The project evaluates whether class imbalance leads to lower macro F1 than weighted F1 and uneven per-class performance. |
| H5 | Validation and test macro F1 are compared to assess whether selected models generalize consistently to unseen test data. The project considers a test macro F1 decrease of no more than 0.10 relative to validation as evidence of consistent generalization. |
| H6 | MobileNetV2 transfer learning is evaluated to determine whether pretrained convolutional features improve validation macro F1 over the CNN baseline, or provide comparable performance within 0.05 macro F1 while using fewer training epochs. |
| H7 | Post-training quantization is evaluated to determine whether MobileNetV2 deployment cost can be reduced while preserving test macro F1 within the project's acceptable tolerance. Notebook 07 treats a macro F1 drop larger than 0.05 as unacceptable. |

## Project Workflow

```text
01 Dataset Exploration
↓
02 Audio Preprocessing
↓
03 Feature Extraction
↓
04 Classical Machine Learning
↓
05 CNN Baseline
↓
06 MobileNetV2 Transfer Learning
↓
07 Quantization
```

**01 Dataset Exploration** inspects GTZAN folders, FMA Medium metadata, class distributions, durations, waveform examples, frequency spectra, Mel Spectrograms, and MFCC representations. It motivates the shared-label strategy and the need for preprocessing.

**02 Audio Preprocessing** loads and harmonizes GTZAN and FMA Medium metadata, filters to seven shared genres, applies duration constraints, combines the datasets, creates stratified train, validation, and test metadata splits, and saves split CSV files under `data/splits/`.

**03 Feature Extraction** loads the saved metadata splits and generates reusable numerical representations. It saves MFCC CSV files under `data/processed/mfcc/` and Mel Spectrogram tensors plus label arrays under `data/processed/mel/`.

**04 Classical Machine Learning** trains SVM and Random Forest models using MFCC summary vectors. It evaluates train and validation performance, selects the best model using validation macro F1, evaluates once on the test split, and saves model and evaluation artifacts.

**05 CNN Baseline** loads saved Mel Spectrogram tensors, trains a small PyTorch CNN from scratch, evaluates validation and test performance, and saves CNN weights, label encoder, training history, reports, metrics, and confusion matrix outputs.

**06 MobileNetV2 Transfer Learning** loads the same Mel Spectrogram tensors, adapts ImageNet-pretrained MobileNetV2 to the seven-genre task, trains the classifier head with the feature extractor frozen, evaluates against the CNN baseline, and saves MobileNetV2 artifacts.

**07 Quantization** loads the saved MobileNetV2 model, evaluates the FP32 baseline, builds static INT8, dynamic Linear-only, and classifier-only dynamic deployment variants, compares size and CPU inference time, and reports whether quantization preserves macro F1.

## Notebook Index

- [01_dataset_exploration.ipynb](01_dataset_exploration.ipynb)
- [02_audio_preprocessing.ipynb](02_audio_preprocessing.ipynb)
- [03_feature_extraction.ipynb](03_feature_extraction.ipynb)
- [04_classical_ml_models.ipynb](04_classical_ml_models.ipynb)
- [05_cnn_baseline.ipynb](05_cnn_baseline.ipynb)
- [06_mobilenetv2_transfer_learning.ipynb](06_mobilenetv2_transfer_learning.ipynb)
- [07_quantization.ipynb](07_quantization.ipynb)

## Repository Structure

- `notebooks/` contains the end-to-end experimental workflow, from dataset exploration through quantization.
- `src/` contains reusable project code for dataset loading, preprocessing, feature extraction, model definitions, training, evaluation, and optimization.
- `tests/` contains lightweight unit tests for preprocessing and feature extraction utilities.
- `docs/` contains project documentation such as methodology, proposal, and references.
- `data/` stores local raw datasets, metadata splits, and processed feature artifacts. It is excluded from Git because it contains large and generated files.
- `models/` stores trained model weights, label encoders, classical model artifacts, and deployment variants. It is excluded from Git.
- `outputs/` stores generated metrics, reports, training histories, quantization comparisons, and confusion matrices. It is excluded from Git.

## Reproducibility

Generated artifacts are excluded from Git to keep the repository lightweight and avoid committing large data, model, and output files. The raw GTZAN and FMA Medium datasets must be downloaded separately and placed in the expected `data/raw/` directory structure before running the workflow.

The notebooks should be executed sequentially because later stages depend on saved artifacts from earlier stages. Notebook 02 creates metadata splits, Notebook 03 creates feature datasets, Notebooks 04 to 06 create model artifacts, and Notebook 07 uses the saved MobileNetV2 model for deployment optimization.

Most generated artifacts are recreated automatically when the notebooks are run in order. Paths, preprocessing parameters, split locations, model locations, and output paths are centralized in `src/utils/config.py`, which helps keep the workflow consistent across notebooks.

## Expected Outputs

| Notebook | Expected Outputs |
|---|---|
| 01 Dataset Exploration | Exploratory tables and visualizations describing GTZAN, FMA Medium, audio durations, waveform structure, frequency content, Mel Spectrograms, MFCCs, and shared-genre strategy. |
| 02 Audio Preprocessing | Train, validation, and test metadata CSV files saved under `data/splits/`. |
| 03 Feature Extraction | MFCC CSV datasets saved under `data/processed/mfcc/`; Mel Spectrogram tensors and label arrays saved under `data/processed/mel/`. |
| 04 Classical Machine Learning | Best classical model, label encoder, optional scaler, metrics summaries, classification reports, and test confusion matrix under `models/classical_ml/` and `outputs/classical_ml/`. |
| 05 CNN Baseline | CNN model weights, CNN label encoder, training history, metrics summary, test classification report, and test confusion matrix under `models/cnn/` and `outputs/cnn/`. |
| 06 MobileNetV2 Transfer Learning | MobileNetV2 model weights, MobileNetV2 label encoder, training history, metrics summary, test classification report, and test confusion matrix under `models/mobilenet/` and `outputs/mobilenet/`. |
| 07 Quantization | FP32 and quantized MobileNetV2 deployment artifacts under `models/quantized/`; quantization metrics, classification reports, model-size comparison, and CPU inference-speed comparison under `outputs/mobilenet/`. |